In [1]:
import gzip, json, itertools, os

path = "./outputs/box0_positive_pairs.json.gz"
assert os.path.exists(path), "Output file not found"

with gzip.open(path, "rt") as f:
    for line in itertools.islice(f, 5):
        pair = json.loads(line)
        print(pair[0])
        print(pair[1])
        print("---")

[COL] givenname [VAL] becmy [COL] surname [VAL] spear [COL] postcode [VAL] 27q28 [COL] suburb [VAL] creswell
[COL] givenname [VAL] becky [COL] surname [VAL] spear [COL] postcode [VAL] 27928.0 [COL] suburb [VAL] creswell
---
[COL] givenname [VAL] tiffamy [COL] surname [VAL] stamper [COL] postcode [VAL] 28906 [COL] suburb [VAL] mufphy
[COL] givenname [VAL] tiffany [COL] surname [VAL] stamper [COL] postcode [VAL] 28906.0 [COL] suburb [VAL] murphy
---
[COL] givenname [VAL] tiffamy [COL] surname [VAL] stamper [COL] postcode [VAL] 28906 [COL] suburb [VAL] mufphy
[COL] givenname [VAL] tiffany [COL] surname [VAL] stamper [COL] postcode [VAL] 28906.0 [COL] suburb [VAL] murphy
---
[COL] givenname [VAL] tiffamy [COL] surname [VAL] stamper [COL] postcode [VAL] 28906 [COL] suburb [VAL] mufphy
[COL] givenname [VAL] tiffany [COL] surname [VAL] stamper [COL] postcode [VAL] 28906.0 [COL] suburb [VAL] murphy
---
[COL] givenname [VAL] tiffamy [COL] surname [VAL] stamper [COL] postcode [VAL] 28906 [COL] s

In [2]:
"""
Inspect tokenization of serialized records and marker tokens to verify whether
our markers are treated as single tokens by the selected tokenizer.
"""

from typing import List, Tuple

import pandas as pd

try:
    # Prefer package-style imports when running from project root
    from rl_ncvr.pair_generation import _try_detect_marker_words, _serialize_row
    from rl_ncvr.box_splitting import get_box, DEFAULT_DATA_DIR
except Exception:
    # Fallback when running this notebook directly inside rl_ncvr/
    from pair_generation import _try_detect_marker_words, _serialize_row
    from box_splitting import get_box, DEFAULT_DATA_DIR

from transformers import AutoTokenizer


def inspect_tokens(text: str, tokenizer) -> List[str]:
    """
    Tokenize the provided text with the given tokenizer and print a concise
    report including the token list and token count. Returns the token list.
    """
    tokens = tokenizer.tokenize(text)
    print(text)
    print(tokens)
    print("num_tokens:", len(tokens))
    print("---")
    return tokens


# Configure model and derive markers used in serialization
model_name = "nreimers/MiniLM-L6-H384-uncased"
field_marker, value_marker = _try_detect_marker_words(model_name)
print("Selected markers:", field_marker, value_marker)
print("Bracketed markers:", f"[{field_marker}]", f"[{value_marker}]")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Inspect tokenization of marker variants (with and without brackets)
for sample in [
    f"[{field_marker}]",
    f"[{value_marker}]",
    field_marker,
    value_marker,
    "[COL]",
    "[VAL]",
    "COL",
    "VAL",
]:
    inspect_tokens(sample, tokenizer)

# Build a few serialized samples from box 0 to check full serialization
entity_columns: Tuple[str, ...] = ("givenname", "surname", "postcode", "suburb")
parts = get_box(0, data_dir=DEFAULT_DATA_DIR, recid_column="recid", chunksize=100_000)

serialized_samples: List[str] = []
for df in parts.values():
    if df is None or len(df) == 0:
        continue
    if not all(col in df.columns for col in entity_columns):
        continue
    head = df.head(5)
    for _, row in head.iterrows():
        serialized_samples.append(
            _serialize_row(
                row,
                columns=entity_columns,
                field_marker=field_marker,
                value_marker=value_marker,
            )
        )
    if len(serialized_samples) >= 5:
        break

print(f"Collected {len(serialized_samples)} serialized samples for inspection")
for s in serialized_samples:
    inspect_tokens(s, tokenizer)



Selected markers: COL VAL
Bracketed markers: [COL] [VAL]
[COL]
['[', 'col', ']']
num_tokens: 3
---
[VAL]
['[', 'val', ']']
num_tokens: 3
---
COL
['col']
num_tokens: 1
---
VAL
['val']
num_tokens: 1
---
[COL]
['[', 'col', ']']
num_tokens: 3
---
[VAL]
['[', 'val', ']']
num_tokens: 3
---
COL
['col']
num_tokens: 1
---
VAL
['val']
num_tokens: 1
---
Collected 5 serialized samples for inspection
[COL] givenname [VAL] becmy [COL] surname [VAL] spear [COL] postcode [VAL] 27q28 [COL] suburb [VAL] creswell
['[', 'col', ']', 'given', '##name', '[', 'val', ']', 'be', '##cm', '##y', '[', 'col', ']', 'surname', '[', 'val', ']', 'spear', '[', 'col', ']', 'post', '##code', '[', 'val', ']', '27', '##q', '##28', '[', 'col', ']', 'suburb', '[', 'val', ']', 'cr', '##es', '##well']
num_tokens: 40
---
[COL] givenname [VAL] tiffamy [COL] surname [VAL] stamper [COL] postcode [VAL] 28906 [COL] suburb [VAL] mufphy
['[', 'col', ']', 'given', '##name', '[', 'val', ']', 'ti', '##ffa', '##my', '[', 'col', ']', 'surna

code to generate the training pairs:

python -m rl_ncvr.pair_generation --box_id 0 --output rl_ncvr/outputs/box0_positive_pairs.json.gz --data_dir data/north_carolina_voters

code to run the training config:

python rl_ncvr/train_script_gpu.py --model nreimers/MiniLM-L6-H384-uncased --steps 2000 --batch_size 64 --data_folder rl_ncvr/outputs rl_ncvr/outputs/data_config.json outputs/run_minilm_box0_gpu